# Component-Level Testing

Deep-dive into individual components with comprehensive train-test evaluation.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from featransform.utils.data_generator import DatasetGenerator
from featransform.processing.imputation import CoreImputer
from featransform.processing.encoding import CategoricalEncoder
from featransform.features.anomaly import AnomalyEnsemble, IsolationForestStrategy
from featransform.features.clustering import ClusteringEnsemble, KMeansStrategy
from featransform.features.dimensionality import DimensionalityEnsemble, PCAStrategy
from featransform.optimization.selector import FeatureSelector
from featransform.core.enums import ImputationStrategy, EncodingStrategy, SelectionStrategy

## Generate Dataset

In [ ]:
X_full, y_full = DatasetGenerator.complex_dataset(n_samples=10000, task='binary_classification')
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(X_full, y_full, test_size=0.3, random_state=42)

## Encoding

In [ ]:
encoder = CategoricalEncoder(strategy=EncodingStrategy.LABEL)
encoder.fit(X_train_full, y_train_full)
X_train_proc = encoder.transform(X_train_full)
X_test_proc = encoder.transform(X_test_full)

## Imputation

In [ ]:
imputer = CoreImputer(strategy=ImputationStrategy.ITERATIVE)
imputer.fit(X_train_proc)
X_train_proc = imputer.transform(X_train_proc)
X_test_proc = imputer.transform(X_test_proc)

## Feature Engineering

In [ ]:
anomaly_ens = AnomalyEnsemble(strategies=[IsolationForestStrategy(n_estimators=100, contamination=0.1)], verbose=False)
anomaly_ens.fit(X_train_proc)
train_anom = anomaly_ens.transform(X_train_proc).features
test_anom = anomaly_ens.transform(X_test_proc).features

cluster_ens = ClusteringEnsemble(strategies=[KMeansStrategy(n_clusters=3, random_state=42)], verbose=False)
cluster_ens.fit(X_train_proc)
train_clust = cluster_ens.transform(X_train_proc).features
test_clust = cluster_ens.transform(X_test_proc).features

dim_ens = DimensionalityEnsemble(strategies=[PCAStrategy(n_components=5)], verbose=False)
dim_ens.fit(X_train_proc, y_train_full)
train_dim = dim_ens.transform(X_train_proc).features
test_dim = dim_ens.transform(X_test_proc).features

## Combine Features

In [ ]:
X_train_all = pd.concat([X_train_proc, train_anom, train_clust, train_dim], axis=1)
X_test_all = pd.concat([X_test_proc, test_anom, test_clust, test_dim], axis=1)

## Feature Selection

In [ ]:
selector = FeatureSelector(strategy=SelectionStrategy.IMPORTANCE, min_features=10, verbose=False)
selector.fit(X_train_all, y_train_full)
selected_features = selector.select_by_threshold(0.7)
selector.set_selected_features(selected_features)
X_train_final = selector.transform(X_train_all)
X_test_final = selector.transform(X_test_all)

scores = selector.get_feature_scores()
if scores:
    top_10 = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:10]
    for feat, score in top_10:
        print(f"{feat}: {score:.4f}")